# EDA — Camada Silver (Iceberg)

A Silver real do lakehouse não é mais Parquet solto — é um conjunto de tabelas **Apache Iceberg** (`lakehouse.silver.*` no Spark, expostas como `iceberg.silver.*` no Trino, mesmo catálogo Hive Metastore). Este notebook consulta essas tabelas via **Trino**, o mesmo motor que serve a Gold — sem precisar subir uma SparkSession só para explorar dado.

Pré-requisito: stack do lakehouse no ar (`hive-metastore`, `trino`, tabelas já escritas pelo `silver_job.py`). Ajuste `TRINO_HOST`/`TRINO_PORT` no `.env` conforme de onde o notebook roda (ver comentário no `.env.example`).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import trino
from dotenv import load_dotenv
import os

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
load_dotenv(dotenv_path=project_root / ".env")

pd.set_option("display.max_columns", 40)

conn = trino.dbapi.connect(
    host=os.environ.get("TRINO_HOST", "trino"),
    port=int(os.environ.get("TRINO_PORT", 8080)),
    user=os.environ.get("TRINO_USER", "notebook"),
    http_scheme=os.environ.get("TRINO_HTTP_SCHEME", "http"),
    catalog=os.environ.get("TRINO_CATALOG", "iceberg"),
    schema="silver",
)


def query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [c[0] for c in cur.description]
    return pd.DataFrame(rows, columns=cols)


query("SHOW TABLES FROM iceberg.silver")


## 1. Volume por tabela


In [ ]:
tabelas = ["empenhos", "ordem_bancaria_orcamentaria", "contratos", "unidade_gestora"]
contagens = query(
    " UNION ALL ".join(
        f"SELECT '{t}' AS tabela, COUNT(*) AS registros FROM iceberg.silver.{t}" for t in tabelas
    )
)
contagens


## 2. Dedup entre execuções — a promessa do `MERGE INTO`

Na versão pandas antiga, dedup só valia *dentro* de uma execução. Aqui, `COUNT(*)` e `COUNT(DISTINCT chave_de_negócio)` devem bater — se não baterem, o `MERGE INTO` não está deduplicando como deveria.


In [ ]:
query(
    """
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT CAST(id AS VARCHAR) || '-' || CAST(ano AS VARCHAR)) AS chaves_distintas
    FROM iceberg.silver.empenhos
    """
)


## 3. Qualidade — nulos nas colunas centrais e cobertura de datas


In [ ]:
query(
    """
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE dataemissao IS NULL) AS sem_data,
        COUNT(*) FILTER (WHERE valor IS NULL) AS sem_valor,
        MIN(dataemissao) AS data_min,
        MAX(dataemissao) AS data_max
    FROM iceberg.silver.empenhos
    """
)


In [ ]:
query(
    """
    SELECT tipo_credor, COUNT(*) AS contratos
    FROM iceberg.silver.contratos
    GROUP BY tipo_credor
    ORDER BY contratos DESC
    """
)


## 4. Time travel — histórico de snapshots

Vantagem direta de Iceberg sobre Parquet solto: cada `MERGE INTO` gera um snapshot novo, consultável pela tabela de metadados `"<tabela>$snapshots"`.


In [ ]:
query(
    """
    SELECT snapshot_id, committed_at, operation
    FROM iceberg.silver."empenhos$snapshots"
    ORDER BY committed_at DESC
    LIMIT 10
    """
)


## Achados rápidos

- Se `total_linhas` == `chaves_distintas` em `empenhos`, o `MERGE INTO` está deduplicando entre execuções — a limitação da Silver pandas antiga não existe mais aqui.
- `"$snapshots"` é o que dá a auditabilidade (time travel) mencionada no storytelling do projeto — dá para provar como a tabela estava numa data específica.
- Próximo passo: `eda_gold.ipynb`, para ver o mesmo dado já modelado em estrela.
